# GeoAI Land Cover Classifier — 04 · Análisis de resultados de validación

**Dataset:** EuroSAT RGB (Sentinel-2) · **Entorno:** Google Colab · **Estado:** experimentos exploratorios.

**Ejecución:** `Entorno de ejecución → Ejecutar todo` desde una sesión nueva. Cada notebook prepara las variables que necesita; no depende de otro notebook abierto.

**Datos:** `./data` se descarga temporalmente en Colab. **Nunca utilices el conjunto test para escoger modelos.**

**Reproducibilidad:** `random_state=42`; pueden existir diferencias de tiempo o pequeñas variaciones entre versiones de dependencias.


## Alcance

Este cuaderno **no evalúa todavía sobre el conjunto test**. Explora y visualiza el registro de experimentos ya publicados. La evaluación final independiente se implementará después de cerrar el procedimiento de selección y entrenamiento.

Este notebook no necesita entrenar modelos ni ejecutar los notebooks 01–03.

In [ ]:
# ============================================================
# LECTURA REPRODUCIBLE DEL HISTORIAL DE EXPERIMENTOS
# ============================================================
# El CSV está publicado en GitHub; pandas puede leerlo por HTTPS.
# Si trabajas sin conexión, sustituye la URL por la ruta local
# "results/experiments.csv" (desde la raíz del repositorio).
import pandas as pd

CSV_URL = (
    "https://raw.githubusercontent.com/Sppgg/"
    "geoai-land-cover-classifier/main/results/experiments.csv"
)

registro = pd.read_csv(CSV_URL)
columnas = {"modelo", "representacion", "alpha", "escalado", "accuracy_val", "macro_f1_val"}
assert columnas.issubset(registro.columns), "Faltan columnas esperadas en el CSV"
assert registro["macro_f1_val"].between(0, 1).all(), "Macro-F1 fuera del rango válido"

display(registro.sort_values("macro_f1_val", ascending=False))

In [ ]:
# ============================================================
# COMPARAR CONFIGURACIONES SIN VOLVER A ENTRENAR
# ============================================================
# La métrica objetivo seleccionada es macro-F1 de validación;
# da el mismo peso a cada categoría.
import matplotlib.pyplot as plt

comparacion = registro.sort_values("macro_f1_val", ascending=True).copy()
comparacion["etiqueta"] = (
    comparacion["representacion"]
    + " | alpha=" + comparacion["alpha"].astype(str)
    + " | scaler=" + comparacion["escalado"].astype(str)
)

fig, ax = plt.subplots(figsize=(11, 5))
ax.barh(comparacion["etiqueta"], comparacion["macro_f1_val"])
ax.set_xlabel("Macro-F1 (validación, muestra de 1.000 imágenes)")
ax.set_title("Experimentos EuroSAT: comparación exploratoria")
ax.set_xlim(0, 1)
plt.tight_layout()
plt.show()

mejor = registro.sort_values("macro_f1_val", ascending=False).iloc[0]
print("Mayor macro-F1 observado:", round(float(mejor["macro_f1_val"]), 4))
print("Configuración:", mejor["representacion"], "alpha=", mejor["alpha"])
print("ATENCIÓN: todavía NO se ha evaluado el modelo en test.")

## Limitaciones y próximos pasos

- La selección de hiperparámetros se basó repetidamente en las mismas 1.000 imágenes de validación: riesgo de sobreajuste a la validación.
- Existe una brecha elevada entre entrenamiento y validación: investigar sobreajuste y nuevas representaciones.
- No hay un *split* geográfico: los resultados no demuestran transferencia a regiones independientes.
- El CSV inicial se compiló manualmente de ejecuciones anteriores, **no es un registro generado automáticamente**; una siguiente mejora será exportar directamente desde el pipeline.
- El test final permanece reservado hasta fijar modelo, preprocesamiento y metodología.

**Nota:** el notebook 03 permite reproducir el historial de entrenamiento con los datos y código originales.